# Preprocesado del Dataset de Salarios

El presente notebook recoge el proceso completo de preparación de datos previo al entrenamiento de modelos de aprendizaje supervisado. El objetivo final es predecir si una persona tiene un salario superior o inferior a 50.000$ anuales a partir de características sociodemográficas y laborales.

El preprocesado se organiza en las siguientes etapas:

1. **Limpieza de datos**: detección y tratamiento de valores faltantes y duplicados.
2. **Transformación y codificación**: manejo de variables categóricas, numéricas y binarias para que sean aptas para su uso en modelos de ML.
3. **Evaluación de métodos de selección de características**: análisis comparativo de los tres enfoques vistos en la asignatura (filtro, wrapper y embebido).
4. **División train / validación / test**: separación de conjuntos garantizando la ausencia de data leakage(que no se entrene el modelo con datos de test).

Cada decisión tomada se justifica en detalle mediante celdas de markdown.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing

## Exploración inicial del dataset

Antes de cualquier transformación, inspeccionamos el dataset para entender su estructura. El archivo `salario.csv` contiene **13 columnas** (12 características + etiqueta objetivo) con información sociodemográfica y laboral de cada persona:

| Columna | Tipo | Descripción |
|---|---|---|
| `edad` | int64 | Edad de la persona |
| `origen_trabajo` | object | Sector del empleador (privado, autónomo, gobierno...) |
| `estudios` | object | Nivel de estudios alcanzado |
| `estado_civil` | object | Situación civil actual |
| `trabajo` | object | Ocupación concreta de la persona |
| `posicion_familiar` | object | Rol en el núcleo familiar |
| `etnia` | object | Origen étnico |
| `sexo` | object | Hombre / Mujer |
| `ganancias_inversiones` | int64 | Ingresos por inversiones |
| `perdidas_inversiones` | int64 | Pérdidas por inversiones |
| `horas_trabajo_semana` | int64 | Horas semanales trabajadas |
| `pais_origen` | object | País de origen |
| `salario` | object | **Etiqueta**: `<=50K` o `>50K` |

A continuación se muestran 5 filas de muestra del dataset antes de cualquier transformación.

In [2]:
cols = ['edad','origen_trabajo','estudios','estado_civil','trabajo','posicion_familiar','etnia','sexo','ganancias_inversiones','perdidas_inversiones','horas_trabajo_semana','pais_origen','salario']
df = pd.read_csv(r'salario.csv',names=cols,header=0)
df.sample(5)

,edad,origen_trabajo,estudios,estado_civil,trabajo,posicion_familiar,etnia,sexo,ganancias_inversiones,perdidas_inversiones,horas_trabajo_semana,pais_origen,salario
27097,31,Private,Assoc-voc,Separated,Tech-support,Soltero-a,Negro,Mujer,0,0,32,United-States,<=50K
26054,43,Private,HS-grad,Divorced,Tech-support,Soltero-a,Negro,Mujer,0,0,35,United-States,<=50K
14805,23,Private,Some-college,Never-married,Sales,Hijo-unico,Blanco,Mujer,0,0,12,United-States,<=50K
22820,46,?,HS-grad,Divorced,?,No-en-familia,Negro,Mujer,0,0,40,United-States,<=50K
27958,31,Private,Bachelors,Never-married,Sales,No-en-familia,Blanco,Hombre,0,0,40,United-States,<=50K


In [3]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27998 entries, 0 to 27997
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   edad                   27998 non-null  int64 
 1   origen_trabajo         27998 non-null  object
 2   estudios               27998 non-null  object
 3   estado_civil           27998 non-null  object
 4   trabajo                27998 non-null  object
 5   posicion_familiar      27998 non-null  object
 6   etnia                  27998 non-null  object
 7   sexo                   27998 non-null  object
 8   ganancias_inversiones  27998 non-null  int64 
 9   perdidas_inversiones   27998 non-null  int64 
 10  horas_trabajo_semana   27998 non-null  int64 
 11  pais_origen            27998 non-null  object
 12  salario                27998 non-null  object
dtypes: int64(4), object(9)
memory usage: 15.0 MB


In [4]:
df.salario.value_counts()

salario
<=50K    21290
>50K      6708
Name: count, dtype: int64

Existe un **desbalanceo moderado** (~75% `<=50K` / ~25% `>50K`). Este dato es relevante porque 
condiciona tanto la elección de métricas (el accuracy no es suficiente) como la necesidad de 
aplicar técnicas de resampling.

Sin embargo, el resampling **no puede aplicarse en este punto**: debe realizarse exclusivamente 
sobre el conjunto de entrenamiento, después de la división train/val/test, para evitar que 
instancias sintéticas generadas a partir de datos de test contaminen el entrenamiento. 
Su tratamiento queda por tanto en `supervised.ipynb`.

## 1.1 Limpieza de datos

El preprocesado de datos consiste en transformar los datos brutos en una representación adecuada para que los algoritmos de ML puedan extraer patrones de forma efectiva. Incluye tres grandes bloques:

- **Limpieza**: detección de valores faltantes, inconsistencias y duplicados.
- **Codificación y transformación**: conversión de variables categóricas a representaciones numéricas y escalado de variables continuas.
- **Selección de características**: identificación de las variables más informativas para la tarea objetivo.

Comenzamos por la limpieza, que es siempre el primer paso antes de cualquier transformación.

In [ ]:
# Los valores faltantes no aparecen como NaN sino codificados como ' ?'
# Los sustituimos por NaN para poder trabajar con ellos con pandas
df.loc[df['origen_trabajo'] == ' ?','origen_trabajo'] = np.nan
df.loc[df['estudios'] == ' ?','estudios'] = np.nan
df.loc[df['estado_civil'] == ' ?','estado_civil'] = np.nan
df.loc[df['trabajo'] == ' ?','trabajo'] = np.nan
df.loc[df['posicion_familiar'] == ' ?','posicion_familiar'] = np.nan
df.loc[df['etnia'] == ' ?','etnia'] = np.nan
df.loc[df['sexo'] == ' ?','sexo'] = np.nan
df.loc[df['pais_origen'] == ' ?','pais_origen'] = np.nan
df.loc[df['salario'] == ' ?','salario'] = np.nan

In [270]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27998 entries, 0 to 27997
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   edad                   27998 non-null  int64 
 1   origen_trabajo         26430 non-null  object
 2   estudios               27998 non-null  object
 3   estado_civil           27998 non-null  object
 4   trabajo                26425 non-null  object
 5   posicion_familiar      27998 non-null  object
 6   etnia                  27998 non-null  object
 7   sexo                   27998 non-null  object
 8   ganancias_inversiones  27998 non-null  int64 
 9   perdidas_inversiones   27998 non-null  int64 
 10  horas_trabajo_semana   27998 non-null  int64 
 11  pais_origen            27503 non-null  object
 12  salario                27998 non-null  object
dtypes: int64(4), object(9)
memory usage: 2.8+ MB


Tras sustituir los `?` por `NaN`, observamos que los valores faltantes afectan a tres columnas: `origen_trabajo`, `trabajo` y `pais_origen`, representando aproximadamente un **6% del total de instancias** en cada caso.

Dado que el porcentaje es reducido, descartar las filas afectadas implicaría perder información válida del resto de columnas. Al tratarse de variables **categóricas**, las estrategias disponibles son:

- **Imputación por moda** (valor más frecuente): apropiada cuando una categoría domina claramente la distribución.
- **Categoría "Unknown"**: apropiada cuando ninguna categoría destaca y no queremos introducir sesgo.

Analizamos la distribución de cada columna para decidir cuál aplicar en cada caso.

In [271]:
df.origen_trabajo.isnull().sum()

np.int64(1568)

In [272]:
df.origen_trabajo.value_counts()

origen_trabajo
Private             19483
Self-emp-not-inc     2211
Local-gov            1810
State-gov            1119
Self-emp-inc          961
Federal-gov           830
Without-pay            11
Never-worked            5
Name: count, dtype: int64

In [273]:
df.trabajo.isnull().sum()

np.int64(1573)

In [274]:
df.trabajo.value_counts()

trabajo
Prof-specialty       3566
Craft-repair         3512
Exec-managerial      3464
Adm-clerical         3305
Sales                3148
Other-service        2857
Machine-op-inspct    1712
Transport-moving     1372
Handlers-cleaners    1146
Farming-fishing       859
Tech-support          791
Protective-serv       552
Priv-house-serv       133
Armed-Forces            8
Name: count, dtype: int64

In [275]:
df.pais_origen.isnull().sum()

np.int64(495)

In [276]:
df.pais_origen.value_counts()

pais_origen
United-States                 25092
Mexico                          554
Philippines                     165
Germany                         116
Canada                          107
Puerto-Rico                     105
El-Salvador                      85
England                          85
Cuba                             82
India                            82
South                            74
China                            67
Jamaica                          66
Vietnam                          58
Italy                            58
Dominican-Republic               57
Guatemala                        56
Japan                            55
Poland                           54
Columbia                         50
Taiwan                           47
Iran                             40
Haiti                            38
Nicaragua                        30
Portugal                         30
Greece                           26
Peru                             25
France          

Tras analizar las distribuciones:

- **`origen_trabajo`**: la categoría `Private` concentra aproximadamente el 70% de los casos → imputamos con la **moda** (`Private`), ya que introducir ese valor no distorsiona la distribución real.
- **`pais_origen`**: `United-States` es ampliamente mayoritario → imputamos igualmente con la **moda**.
- **`trabajo`**: ninguna categoría domina de forma clara → imputar con la moda introduciría un sesgo artificial. Se opta por asignar la categoría `Unknown`, preservando la fila sin alterar la distribución de las demás variables.

In [277]:
df['origen_trabajo'] = df['origen_trabajo'].fillna(' Private')
df['trabajo'] = df['trabajo'].fillna(' Unknown')
df['pais_origen'] = df['pais_origen'].fillna(' United-States')


In [278]:
df.pais_origen.value_counts()

pais_origen
United-States                 25587
Mexico                          554
Philippines                     165
Germany                         116
Canada                          107
Puerto-Rico                     105
El-Salvador                      85
England                          85
Cuba                             82
India                            82
South                            74
China                            67
Jamaica                          66
Vietnam                          58
Italy                            58
Dominican-Republic               57
Guatemala                        56
Japan                            55
Poland                           54
Columbia                         50
Taiwan                           47
Iran                             40
Haiti                            38
Nicaragua                        30
Portugal                         30
Greece                           26
Peru                             25
France          

In [ ]:
# Comprobamos si existen filas duplicadas en el dataset
df.duplicated().sum()

np.int64(2777)

In [ ]:
# Eliminamos los duplicados encontrados
df.drop_duplicates(inplace=True)

In [281]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25221 entries, 0 to 27997
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   edad                   25221 non-null  int64 
 1   origen_trabajo         25221 non-null  object
 2   estudios               25221 non-null  object
 3   estado_civil           25221 non-null  object
 4   trabajo                25221 non-null  object
 5   posicion_familiar      25221 non-null  object
 6   etnia                  25221 non-null  object
 7   sexo                   25221 non-null  object
 8   ganancias_inversiones  25221 non-null  int64 
 9   perdidas_inversiones   25221 non-null  int64 
 10  horas_trabajo_semana   25221 non-null  int64 
 11  pais_origen            25221 non-null  object
 12  salario                25221 non-null  object
dtypes: int64(4), object(9)
memory usage: 2.7+ MB


## 1.2 Transformación y codificación de variables

Codificamos cada columna en función de su naturaleza. El criterio general es:

- **Variables continuas** (`edad`, `horas_trabajo_semana`): normalización para evitar que la escala distorsione el aprendizaje.
- **Variables ordinales** (`estudios`, `ganancias_inversiones`, `perdidas_inversiones`): binning o encoding respetando el orden natural.
- **Variables categóricas nominales** (`origen_trabajo`, `estado_civil`, `trabajo`, etc.): One-Hot Encoding, definiendo explícitamente el conjunto de categorías para que sea robusto ante nuevas categorías en el conjunto de test.
- **Variable binaria** (`sexo`): mapeo directo a 0/1.
- **Etiqueta** (`salario`): mapeo a 0/1.

In [282]:
from sklearn.preprocessing import StandardScaler
#edad
#version para binning
#bins = list(range(0,110,10))
#df.edad = pd.get_dummies(pd.cut(df.edad,bins=bins,right=False),dtype=int).values.tolist()
#version estandarizada
#df.edad = (df.edad - df.edad.mean())/df.edad.std()
#version normalizar, no se usa el max min Scaler por si en el test aparecen nuevos min/max, así tener la misma escala 
df.edad = df.edad/100
#origen_trabajo
orden_origen_trabajo = [' Private', ' Self-emp-not-inc', ' Self-emp-inc', ' Federal-gov',
         ' Local-gov', ' State-gov', ' Without-pay', ' Never-worked']
df['origen_trabajo'] = pd.get_dummies(pd.Categorical(df['origen_trabajo'],categories=orden_origen_trabajo),dtype=int).values.tolist()

#estudios
orden_estudios = [' Preschool', ' 1st-4th', ' 5th-6th', ' 7th-8th', ' 9th',
                  ' 10th', ' 11th', ' 12th', ' HS-grad', ' Some-college',
                  ' Assoc-acdm', ' Assoc-voc', ' Bachelors', ' Masters',
                  ' Prof-school', ' Doctorate']
df['estudios'] = pd.get_dummies(pd.Categorical(df['estudios'],categories=orden_estudios,ordered=True),dtype=int).values.tolist()

#estado_civil
orden_estado_civil = [' Never-married', ' Married-civ-spouse', ' Divorced',
 ' Married-spouse-absent', ' Separated', ' Married-AF-spouse', ' Widowed']
df['estado_civil'] = pd.get_dummies(pd.Categorical(df['estado_civil'],categories=orden_estado_civil),dtype=int).values.tolist()


#trabajo 
orden_trabajo = [     ' Adm-clerical',   ' Exec-managerial', ' Handlers-cleaners',
    ' Prof-specialty',     ' Other-service',             ' Sales',
  ' Transport-moving',   ' Farming-fishing', ' Machine-op-inspct',
      ' Tech-support',      ' Craft-repair',   ' Protective-serv',
      ' Armed-Forces',   ' Priv-house-serv',' Unknown']
df['trabajo'] = pd.get_dummies(pd.Categorical(df['trabajo'],categories=orden_trabajo),dtype=int).values.tolist()

#posicion_familiar
orden_familiar = [' No-en-familia',        ' Marido',         ' Mujer',    ' Hijo-unico',
     ' Soltero-a', ' Otro-familiar']
df['posicion_familiar'] = pd.get_dummies(pd.Categorical(df['posicion_familiar'],categories=orden_familiar),dtype=int).values.tolist()


#etnia
orden_etnia = [' Blanco',                    ' Negro',
 ' Asiatico-Pacifico-Isleno',       ' Amerindio-esquimal',
                     ' Otro']
df['etnia'] = pd.get_dummies(pd.Categorical(df['etnia'],categories=orden_etnia),dtype=int).values.tolist()

#sexo
df.sexo = df["sexo"].map({" Hombre": 0, " Mujer": 1})

# ganancias
bins_gan = [-1, 0, 5000,99998, np.inf]
labels_gan = [0, 1, 2,3]
df['ganancias_inversiones'] = pd.cut(
    df['ganancias_inversiones'],
    bins=bins_gan,
    labels=labels_gan
).astype(int)

# perdidas
bins_per = [-1, 0, 1900, np.inf]
labels_per = [0, 1, 2]
df['perdidas_inversiones'] = pd.cut(
    df['perdidas_inversiones'],
    bins=bins_per,
    labels=labels_per
).astype(int)


#horas trabajo semanales
scaler_horas = StandardScaler()
df['horas_trabajo_semana'] = scaler_horas.fit_transform(df[['horas_trabajo_semana']])


#pais origen
orden_pais = [             ' United-States',                       ' Cuba',
                    ' Jamaica',                      ' India',
                     ' Mexico',                ' Puerto-Rico',
                   ' Honduras',                    ' England',
                     ' Canada',                    ' Germany',
                       ' Iran',                ' Philippines',
                     ' Poland',                   ' Columbia',
                   ' Cambodia',                   ' Thailand',
                    ' Ecuador',                       ' Laos',
                     ' Taiwan',                      ' Haiti',
                   ' Portugal',         ' Dominican-Republic',
                ' El-Salvador',                     ' France',
                  ' Guatemala',                      ' Italy',
                      ' China',                      ' South',
                      ' Japan',                 ' Yugoslavia',
                       ' Peru', ' Outlying-US(Guam-USVI-etc)',
                   ' Scotland',            ' Trinadad&Tobago',
                     ' Greece',                  ' Nicaragua',
                    ' Vietnam',                       ' Hong',
                    ' Ireland',                    ' Hungary',
         ' Holand-Netherlands']
df['pais_origen'] = pd.get_dummies(pd.Categorical(df['pais_origen'],categories=orden_pais,ordered=True),dtype=int).values.tolist()

# salario 
df.salario = df["salario"].map({' <=50K': 0, ' >50K': 1})

In [283]:
df.sample(5)

,edad,origen_trabajo,estudios,estado_civil,trabajo,posicion_familiar,etnia,sexo,ganancias_inversiones,perdidas_inversiones,horas_trabajo_semana,pais_origen,salario
9556,0.44,"[1, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]","[0, 0, 0, 0, 1, 0, 0]","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 1, 0]","[1, 0, 0, 0, 0]",1,0,0,0.585066,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0
1722,0.38,"[1, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]","[0, 0, 1, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0]",0,0,0,0.585066,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1
24603,0.35,"[0, 1, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]","[0, 1, 0, 0, 0, 0, 0]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 0, 0, 0, 0]","[1, 0, 0, 0, 0]",0,0,0,0.743006,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1
25280,0.32,"[1, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]","[0, 1, 0, 0, 0, 0]","[1, 0, 0, 0, 0]",0,0,0,-0.046691,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1
244,0.38,"[1, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]","[0, 0, 1, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0]",0,0,1,-0.046691,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0


In [284]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25221 entries, 0 to 27997
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   edad                   25221 non-null  float64
 1   origen_trabajo         25221 non-null  object 
 2   estudios               25221 non-null  object 
 3   estado_civil           25221 non-null  object 
 4   trabajo                25221 non-null  object 
 5   posicion_familiar      25221 non-null  object 
 6   etnia                  25221 non-null  object 
 7   sexo                   25221 non-null  int64  
 8   ganancias_inversiones  25221 non-null  int64  
 9   perdidas_inversiones   25221 non-null  int64  
 10  horas_trabajo_semana   25221 non-null  float64
 11  pais_origen            25221 non-null  object 
 12  salario                25221 non-null  int64  
dtypes: float64(2), int64(4), object(7)
memory usage: 2.7+ MB


## 1.3 Evaluación exhaustiva de métodos de selección de características

La selección de características tiene como objetivo reducir la dimensionalidad del dataset 
eliminando atributos irrelevantes o redundantes, mejorando así la capacidad de generalización 
de los modelos y reduciendo el riesgo de overfitting.

Existen tres grandes familias de métodos, cada una con propiedades distintas:

| Método | Cómo funciona | Ventajas | Inconvenientes |
|--------|--------------|----------|----------------|
| **Filtro** | Evalúa cada característica de forma independiente usando métricas estadísticas (correlación, F-score, Chi-square...). No interviene ningún modelo de ML. | Independiente del clasificador. Bajo coste computacional. Buena capacidad de generalización. | No captura interacciones entre características. |
| **Embebido** | La selección ocurre durante el entrenamiento del modelo (ej. LASSO/L1 lleva coeficientes a 0). | Tiene en cuenta la interacción entre características. Más rápido que wrapper. | Dependiente del algoritmo elegido. |
| **Wrapper** | Evalúa subconjuntos de features entrenando un modelo real con cada subconjunto. | Tiene en cuenta la interacción entre características y el clasificador. | Computacionalmente muy costoso. Riesgo de overfitting. Dependiente del clasificador. |

A continuación aplicamos los tres métodos sobre el dataset expandido (`df_expanded`), comparando los subconjuntos resultantes.

> **Nota sobre el orden:** el método filtro se aplica sobre el dataset completo antes de la división (es independiente del modelo y no produce leakage). Los métodos wrapper y embebido, al entrenar modelos internamente, se ajustan **exclusivamente sobre `X_train`** para evitar data leakage.



In [ ]:
# convertimos a columnas boleeanas,porque nos damos cuenta de que nuestro enfoque anterior al plantear el dataframe no era el adecuado, 
# ahora si que estamos listos para su posterior entrenamiento en base al resto de modelos 
import sklearn.feature_selection as fs

def expandir_ohe(df, col, orden, prefijo):
    # La columna ya contiene listas [0,1,0,...], las expandemos directamente
    cols = [f'{prefijo}_{c.strip()}' for c in orden]
    return pd.DataFrame(df[col].tolist(), columns=cols, index=df.index)

df_expanded = pd.concat([
    df[['edad', 'sexo', 'ganancias_inversiones', 'perdidas_inversiones', 'horas_trabajo_semana']],
    expandir_ohe(df, 'origen_trabajo',    orden_origen_trabajo, 'origen_trabajo'),
    expandir_ohe(df, 'estudios',          orden_estudios,       'estudios'),
    expandir_ohe(df, 'estado_civil',      orden_estado_civil,   'estado_civil'),
    expandir_ohe(df, 'trabajo',           orden_trabajo,        'trabajo'),
    expandir_ohe(df, 'posicion_familiar', orden_familiar,       'posicion_familiar'),
    expandir_ohe(df, 'etnia',             orden_etnia,          'etnia'),
    expandir_ohe(df, 'pais_origen',       orden_pais,           'pais_origen'),
], axis=1)

Y = df['salario'].tolist()

### Método Filtro — Pearson, F-score y Chi-square

El método **filtro** evalúa la relevancia de cada característica de forma **independiente** respecto a la etiqueta objetivo, usando criterios estadísticos sin intervención de ningún modelo de ML. Es el método más rápido y generalizable, y su resultado es válido para cualquier clasificador posterior.

Dado que las variables del dataset son de naturaleza heterogénea, aplicamos un test estadístico distinto según el tipo de cada variable:

| Variable | Tipo | Test aplicado | Criterio de selección |
|---|---|---|---|
| `edad`, `horas_trabajo_semana` | Continua | **Pearson** | \|r\| > 0.1 |
| `sexo`, `ganancias_inversiones`, `perdidas_inversiones` | Ordinal / discreta | **F-score** | p-valor < 0.05 |
| Variables One-Hot (categóricas) | Binaria | **Chi-square** | p-valor < 0.05 |

El selector se aplica sobre el dataset completo (`df_expanded`) antes de la división, lo cual es válido porque el filtro no entrena ningún modelo ni aprende ningún patrón predictivo: simplemente mide asociación estadística entre cada variable y la etiqueta.

In [ ]:
#pearson para variables continuas
cols_continuas = ['edad', 'horas_trabajo_semana']

pearson_r = fs.r_regression(df_expanded[cols_continuas], Y)

print('=== PEARSON (variables continuas) ===')
for col, r in zip(cols_continuas, pearson_r):
    print(f'  {col}: r={r:.4f}  (|r| > 0.1 se considera relevante)')

# Criterio de selección: |r| > 0.1 (correlación mínima con la etiqueta)
cols_continuas_sel = [col for col, r in zip(cols_continuas, pearson_r) if abs(r) > 0.1]

#  f_socre para variables ordinales discretas
cols_ordinales = ['sexo', 'ganancias_inversiones', 'perdidas_inversiones']
f_scores, f_pvalues = fs.f_classif(df_expanded[cols_ordinales], Y)

print('\n=== F-SCORE (variables ordinales/discretas) ===')
for col, f, p in zip(cols_ordinales, f_scores, f_pvalues):
    print(f'  {col}: F={f:.2f}, p={p:.2e}')

cols_ordinales_sel = [col for col, p in zip(cols_ordinales, f_pvalues) if p < 0.05]

# chi_square para variables categóricas One-Hot
cols_ohe = [c for c in df_expanded.columns
            if c not in cols_continuas + cols_ordinales]

chi2_scores, chi2_pvalues = fs.chi2(df_expanded[cols_ohe], Y)

print('\n=== CHI-SQUARE (variables categóricas OHE) ===')
resultados_chi2 = pd.DataFrame({
    'feature': cols_ohe,
    'chi2': chi2_scores,
    'p_value': chi2_pvalues
}).sort_values('chi2', ascending=False)
print(resultados_chi2.to_string())

cols_ohe_sel = resultados_chi2[resultados_chi2['p_value'] < 0.05]['feature'].tolist()
#selección final 
features_seleccionadas = cols_continuas_sel + cols_ordinales_sel + cols_ohe_sel

print(f'\n=== FEATURES SELECCIONADAS ({len(features_seleccionadas)}) ===')
print(features_seleccionadas)

df_filtrado = df_expanded[features_seleccionadas]

=== PEARSON (variables continuas) ===
  edad: r=0.2151  (|r| > 0.1 se considera relevante)
  horas_trabajo_semana: r=0.2248  (|r| > 0.1 se considera relevante)

=== F-SCORE (variables ordinales/discretas) ===
  sexo: F=1244.38, p=4.04e-266
  ganancias_inversiones: F=3230.59, p=0.00e+00
  perdidas_inversiones: F=653.35, p=2.71e-142

=== CHI-SQUARE (variables categóricas OHE) ===
                                   feature         chi2        p_value
25         estado_civil_Married-civ-spouse  2731.477189   0.000000e+00
47                posicion_familiar_Marido  2447.871272   0.000000e+00
24              estado_civil_Never-married  1588.039057   0.000000e+00
49            posicion_familiar_Hijo-unico  1017.315530  3.094039e-223
32                 trabajo_Exec-managerial   898.561718  2.015980e-197
34                  trabajo_Prof-specialty   721.364115  6.761383e-159
21                        estudios_Masters   702.446871  8.782483e-155
46         posicion_familiar_No-en-familia   669.16

## División train / validación / test

Antes de aplicar los métodos de selección basados en modelos (wrapper y embebido), dividimos el dataset en tres conjuntos **disjuntos**:

| Conjunto | Proporción | Uso |
|---|---|---|
| `X_train` | 70% | Ajuste del selector + entrenamiento del modelo |
| `X_val` | 15% | Early stopping del modelo (nunca visto por el selector) |
| `X_test` | 15% | Evaluación final, no visto en ningún paso anterior |

**¿Por qué dividir antes de seleccionar?** Los métodos wrapper y embebido aprenden qué features son relevantes entrenando un modelo internamente. Si usaran datos de validación o test en ese proceso, estarían filtrando información de esos conjuntos hacia la selección → **data leakage** → evaluación optimista y no representativa del rendimiento real.

La división se realiza con `shuffle=True` y `random_state=42` para garantizar reproducibilidad.

In [ ]:
from sklearn.model_selection import train_test_split

# Primera división: separar test definitivo (15%)
X_trainval, X_test, Y_trainval, Y_test = train_test_split(
    df_expanded, Y, test_size=0.15, shuffle=True, random_state=42
)

# Segunda división: separar validación del train (15% del total ≈ 18% de trainval)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_trainval, Y_trainval, test_size=0.176, shuffle=True, random_state=42
)

print(f'Train:      {X_train.shape[0]} instancias × {X_train.shape[1]} features')
print(f'Validación: {X_val.shape[0]} instancias × {X_val.shape[1]} features')
print(f'Test:       {X_test.shape[0]} instancias × {X_test.shape[1]} features')

### Método Wrapper — Búsqueda Greedy Bidireccional (BG)

El método **wrapper** evalúa subconjuntos de características entrenando un modelo real con cada subconjunto. Es el enfoque más costoso computacionalmente pero el más informativo porque tiene en cuenta las interacciones entre features y el clasificador concreto.

Implementamos la variante **BG (Bidirectional Generation)** combinando dos búsquedas greedy simultáneas:

- **SFG (Sequential Forward Generation)**: parte del conjunto vacío y añade iterativamente la feature que más mejora la métrica.
- **SBG (Sequential Backward Generation)**: parte del conjunto completo y elimina iterativamente la feature menos informativa.

El subconjunto final es la **unión** de las features seleccionadas por ambas búsquedas, capturando así tanto las más relevantes individualmente (SFG) como las que aportan valor en contexto del conjunto completo (SBG).

El selector se ajusta **únicamente sobre `X_train`** y luego se aplica con `transform` puro sobre `X_val` y `X_test`.

In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import f1_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import make_scorer

scorer_macro = make_scorer(f1_score, average='macro')

rf_wrap = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

# SFG: arranca con 0 features, va añadiendo las mejores
sfs_forward = SequentialFeatureSelector(
    rf_wrap,
    n_features_to_select=30,   # número aproximado — ajustar según tiempo disponible
    direction='forward',
    scoring=scorer_macro,
    cv=3,
    n_jobs=-1
)
sfs_forward.fit(X_train, Y_train)

# SBG: arranca con todas las features, va eliminando las peores
sfs_backward = SequentialFeatureSelector(
    rf_wrap,
    n_features_to_select=30,
    direction='backward',
    scoring=scorer_macro,
    cv=3,
    n_jobs=-1
)
sfs_backward.fit(X_train, Y_train)

# BG: unión de los dos soportes (features seleccionadas por alguno de los dos)
support_bg = sfs_forward.get_support() | sfs_backward.get_support()
features_bg = np.array(X_train.columns)[support_bg]

X_train_bg = X_train.values[:, support_bg]
X_val_bg   = X_val.values[:, support_bg]
X_test_bg  = X_test.values[:, support_bg]

n_bg = X_train_bg.shape[1]
print(f'[WRAPPER BG] Features SFG:  {sfs_forward.get_support().sum()}')
print(f'[WRAPPER BG] Features SBG:  {sfs_backward.get_support().sum()}')
print(f'[WRAPPER BG] Features unión BG: {n_bg} / {X_train.shape[1]}')
print(f'Features: {list(features_bg)}')

#MLP sobre el subconjunto BG
tf.random.set_seed(42)

model_bg = Sequential([
    Input(shape=(n_bg,)),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_bg.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_focal_crossentropy',
    metrics=['accuracy']
)

history_bg = model_bg.fit(
    X_train_bg, Y_train,
    validation_data=(X_val_bg, Y_val.values),
    epochs=100,
    batch_size=64,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10,
                             restore_best_weights=True)],
    verbose=1
)

#Evaluación final
pred_bg = (model_bg.predict(X_test_bg, verbose=0).flatten() > 0.5).astype(int)

print(f'\n[WRAPPER BG] F1-score macro (θ=0.5): {f1_score(Y_test, pred_bg, average="macro"):.4f}')
print(classification_report(Y_test, pred_bg, target_names=['<=50K', '>50K']))

### Método Embebido — SelectFromModel con RandomForestClassifier

Los métodos **embebidos** realizan la selección de características **durante el propio entrenamiento** del modelo, aprovechando información interna del algoritmo (como las importancias de features en árboles de decisión). Son más rápidos que los wrappers y tienen en cuenta las interacciones entre variables, aunque el resultado es específico del algoritmo usado.

Usamos `SelectFromModel` con un `RandomForestClassifier`: el bosque se entrena sobre `X_train` y sus `feature_importances_` determinan qué variables superar el umbral (fijado en la media de importancias). Las features por debajo del umbral se descartan.

Al igual que en el wrapper, el selector se ajusta **solo sobre `X_train`** y se aplica con `transform` puro sobre los demás conjuntos.

In [ ]:
#ajustamos el selector solo a X_train
rf_emb = RandomForestClassifier(
    n_estimators=200,
    max_features='sqrt',
    max_depth=None,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

selector_emb = SelectFromModel(rf_emb, threshold='mean')
selector_emb.fit(X_train, Y_train)   # aprende importancias solo con train

X_train_emb = selector_emb.transform(X_train)
X_val_emb   = selector_emb.transform(X_val)    # transform puro, sin fit
X_test_emb  = selector_emb.transform(X_test)   # transform puro, sin fit

n_emb = X_train_emb.shape[1]
features_emb = np.array(X_train.columns)[selector_emb.get_support()]
print(f'[EMBEBIDO] Features seleccionadas: {n_emb} / {X_train.shape[1]}')
print(f'Features: {list(features_emb)}')

#mlp sobre el subconjunto embebido
tf.random.set_seed(42)

model_emb = Sequential([
    Input(shape=(n_emb,)),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_emb.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_focal_crossentropy',
    metrics=['accuracy']
)

# X_val se usa SOLO para early stopping, no participa en la selección de features
history_emb = model_emb.fit(
    X_train_emb, Y_train,
    validation_data=(X_val_emb, Y_val.values),
    epochs=100,
    batch_size=64,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10,
                             restore_best_weights=True)],
    verbose=1
)

# Evaluación final
pred_emb = (model_emb.predict(X_test_emb, verbose=0).flatten() > 0.5).astype(int)

print(f'\n[EMBEBIDO] F1-score macro (θ=0.5): {f1_score(Y_test, pred_emb, average="macro"):.4f}')
print(classification_report(Y_test, pred_emb, target_names=['<=50K', '>50K']))

## 1.6 Selección del método de características y generación del dataset procesado

Tras evaluar los tres métodos, **se elige el método filtro** como base para el dataset procesado (`processed_def_dataset.csv`) que se usará en `supervised.ipynb`. La justificación es la siguiente:

**¿Por qué filtro y no wrapper o embebido?**

1. **Independencia del clasificador**: el método filtro evalúa la relevancia de cada variable usando criterios estadísticos (Pearson, F-score, Chi-square) sin depender de ningún modelo concreto. Esto produce un dataset genérico válido para todos los clasificadores que se evaluarán en `supervised.ipynb` (regresión logística, SVM, KNN, árboles, redes neuronales...). Un dataset filtrado por un wrapper basado en RandomForest estaría sesgado hacia ese tipo de modelo.

2. **Ausencia de data leakage en la selección**: el filtro se aplica sobre el dataset completo antes de la división, pero sus criterios son puramente estadísticos y no "aprenden" ningún patrón predictivo. No existe riesgo de que la selección esté influida por el conjunto de test.

3. **Generalización**: los métodos filtro tienden a producir subconjuntos con mejor capacidad de generalización porque no se adaptan a ningún clasificador específico, reduciendo el riesgo de overfitting en la selección.

4. **Eficiencia**: dado que el dataset procesado debe ser el punto de partida para múltiples modelos, usar un método computacionalmente ligero e independiente del clasificador es la elección más coherente.


## Resultado del preprocesado

El dataset procesado `processed_dataset.csv` contiene las features seleccionadas por el método filtro junto con la etiqueta `salario`. Este archivo es el punto de partida para el notebook `supervised.ipynb`.

La división en conjuntos train / validación / test se realizará directamente en `supervised.ipynb` sobre este dataset procesado, utilizando los mismos porcentajes (70/15/15) y semilla aleatoria para garantizar la reproducibilidad.

In [ ]:
df_filtrado = df_filtrado.copy()
df_filtrado['salario'] = df['salario'].values  # .values ignora el índice
df_filtrado.to_csv('processed_dataset.csv', sep=',', index=False)

/tmp/ipykernel_6265/3727681575.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado['salario'] = df.salario
